# Can It Notice Its Own Thoughts? Qwen2.5 family on a free Kaggle GPU

Settings: Accelerator = GPU T4, Internet = on (needs a phone-verified account). Add the dataset holding earlier trials (optional) as input; the sweep skips anything already done. Use **Save Version > Save & Run All** so it runs without the browser open; results land in Output.

In [ ]:
import glob, os, shutil
RUN = '/kaggle/working/introspect-runs'
os.makedirs(RUN, exist_ok=True)
# earlier trials from any attached dataset (a zip upload is unpacked by Kaggle)
for d in glob.glob('/kaggle/input/**/qwen-family', recursive=True):
    shutil.copytree(d, f'{RUN}/qwen-family', dirs_exist_ok=True)
    print('resuming from', d)
!find {RUN} -name trials.jsonl | xargs -r wc -l

In [ ]:
!git clone -q https://github.com/feliciachen01/introspect.git /kaggle/working/introspect
%cd /kaggle/working/introspect
!pip install -q -e . bitsandbytes
!pytest -q

In [ ]:
# 1. Generate: concept vectors + injected / control / bias trials (resumes)
!introspect sweep --config configs/kaggle.yaml

In [ ]:
# 2. Grade with vLLM
!pip install -q vllm
!introspect judge --config configs/kaggle.yaml

In [ ]:
# 3. Report
!introspect report --config configs/kaggle.yaml
from IPython.display import Markdown, display
display(Markdown(open('/kaggle/working/introspect-runs/qwen-family/results.md').read()))

In [ ]:
# keep Output small: drop the cloned repo and the large concept-vector files
!rm -rf /kaggle/working/introspect
!find /kaggle/working/introspect-runs -name '*.npz' -delete
!du -sh /kaggle/working/introspect-runs